<a href="https://colab.research.google.com/github/avishek-astra/Deep_Learning_Experiments/blob/main/CNN_codeChallengeNumChans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#IMPORT LIBRARIES
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
import copy
from sklearn.model_selection import train_test_split
#form importing data
import torchvision

import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [ ]:
#use gpu if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

Import and inspect the data

In [ ]:
#download the dataset
cdata=torchvision.datasets.EMNIST(root='emnist',split='letters',download=True)

In [ ]:
#inspect the data
#the categories(but how many letters?)
print(cdata.classes)
print(str(len(cdata.classes))+'Classes')

print('\n Data size:')
print(cdata.data.shape)
#transform to 4D tensor for conv layers(and transform from int8 to float)
images=cdata.data.view([124800,1,28,28]).float()
print('\nTensor data:')
print(images.shape)

In [ ]:
#brief aside:class 'N/A' doesn't exist in the  data.
print(torch.sum(cdata.targets==0))
#however, it causes problem in one-hot encoding
torch.unique(cdata.targets)

In [ ]:
cdata.class_to_idx

In [ ]:
#so therefore we'll eliminate it and subtract 1 from the original
#remove the first class category
letterCategories=cdata.classes[1:]
#relabel label to start to 0
labels=copy.deepcopy(cdata.targets)-1
print(labels.shape)

In [ ]:
##
print(torch.sum(labels==0))
torch.unique(labels)

In [ ]:
# next issue: do we need to normalize the images?
plt.hist(images[:10,:,:,:].view(1,-1).detach(),40);
plt.title('Raw values')
plt.show()

In [ ]:
#yarp
images=images/torch.max(images)
plt.hist(images[:10,:,:,:].view(1,-1).detach(),40);
plt.title('Normalized values')
plt.show()

In [ ]:
#visualize some images
fig,axs=plt.subplots(3,7,figsize=(13,6))
for i,ax in enumerate(axs.flatten()):
  #pick a random pic
  whichpic=np.random.randint(images.shape[0])
  #extract the images and its target letter
  I=np.squeeze(images[whichpic,:,:])
  letter=letterCategories[labels[whichpic]]

  #visualize
  ax.imshow(I.T,cmap='grey')
  ax.set_title("The letter '%s'"%letter)
  ax.set_xticks([])
  ax.set_yticks([])
plt.show()

Create train/test groups using DataLoader

In [ ]:
#step 2: use scikitlearn to split the data
train_data,test_data,train_labels,test_labels=train_test_split(images,labels,test_size=.1)
#step3:convert into pytorch Datasets
train_data=TensorDataset(train_data,train_labels)
test_data=TensorDataset(test_data,test_labels)

#step 4:translate into dataloader objects
batchsize=32
train_loader=DataLoader(train_data,batch_size=batchsize,shuffle=True,drop_last=True)
test_loader=DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])

In [ ]:
#check size (should be images X channels X width X height)
print(train_loader.dataset.tensors[0].shape)
print(train_loader.dataset.tensors[1].shape)

Create train/test groups using Dataloader

In [ ]:
#create a class for the model
def makeTheNet(numchans=(6,6)):
  class emnistnet(nn.Module):
    def __init__(self,numchans):
      super().__init__()
      #self.print=printtoggle # REMOVED: printtoggle is not defined

      ###-------------feature map layers ------------##
      #first convolution layer
      self.conv1=nn.Conv2d(1,numchans[0],3,padding=1)
      self.bnorm1=nn.BatchNorm2d(numchans[0])#input the number of channels in this layer
      #output size (28+2*1-3)/1+1=28/2=14(/2 b/c maxpool)
      #second convolution layer
      self.conv2=nn.Conv2d(numchans[0],numchans[1],3,padding=1)
      self.bnorm2=nn.BatchNorm2d(numchans[1])
      #output size:(14+2*1-3)/1 +1=14/2 =7 (/2 b/c maxpool)

      #-----linear decision layers-------###
      self.fc1=nn.Linear(7*7*numchans[1],50)
      self.fc2=nn.Linear(50,26)

    def forward(self,x):
      x=F.max_pool2d(self.conv1(x),2)
      x=F.leaky_relu(self.bnorm1(x))

      x=F.max_pool2d(self.conv2(x),2)
      x=F.leaky_relu(self.bnorm2(x))



      #reshape for linear layer
      nUnits=x.shape.numel()/x.shape[0]
      x=x.view(-1,int(nUnits))



      #linear layers
      x=F.leaky_relu(self.fc1(x))
      x=self.fc2(x)

      return x
  #create the model instance
  net=emnistnet(numchans)

  #loss fuction
  lossfun=nn.CrossEntropyLoss()

  #optimizer
  optimizer=torch.optim.Adam(net.parameters(),lr=.001)

  return net,lossfun,optimizer

In [ ]:
#test the model with one batch
net,lossfun,optimizer=makeTheNet(((6,12)))

X,y=next(iter(train_loader))
yHat=net(X)

#check size of output
print('\nOutput size:')
print(yHat.shape)

#check loss
loss=lossfun(yHat,torch.squeeze(y))
print('')
print("Loss")
print(loss)

Create a function that trains the model

In [ ]:
#a fuction that trains the model
def function2trainTheModel(numchans):

  #number of epochs
  numepochs=10

  #create a new model
  net,lossfun,optimizer=makeTheNet(numchans)

  #send the model to the CPU
  net.to(device)

  #initialize losses
  trainLoss=np.zeros(numepochs)
  testLoss=np.zeros(numepochs)
  trainErr=np.zeros(numepochs)
  testErr=np.zeros(numepochs)

  #loop over epochs
  for epochi in range(numepochs):

    #loop over training data batches
    net.train()
    batchLoss=[]
    batchErr=[]
    for X,y in train_loader:
      #push data to GPU
      X=X.to(device)
      y=y.to(device)

      #forward pass and loss
      yHat=net(X)
      loss=lossfun(yHat,y)


      #backprop
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      #loss and error from this batch
      batchLoss.append(loss.item())
      batchErr.append(torch.mean((torch.argmax(yHat,axis=1)!=y).float()).item())
    #end of batch loop...

    #and get average losses and error rates across the batches
    trainLoss[epochi]=np.mean(batchLoss)
    trainErr[epochi]=100*np.mean(batchErr)

    ###test performace
    net.eval()
    X,y=next(iter(test_loader)) #extract X,y from test dataLoader

    #push data to CPU
    X=X.to(device)
    y=y.to(device)

    with torch.no_grad(): #deactivates autograd
      yHat=net(X)
      loss=lossfun(yHat,y)

    testLoss[epochi]=loss.item()
    testErr[epochi]=100*torch.mean((torch.argmax(yHat,axis=1)!=y).float()).item()
  #end epochs

#function output
  return trainLoss,testLoss,trainErr,testErr,net

Run the model and show the results!

In [ ]:
#~ 2 minutes with 10 epochs on GPU
trainLoss,testLoss,trainErr,testErr,net=function2trainTheModel((3,7))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(16,5))

ax[0].plot(trainLoss,'s-',label='Train')
ax[0].plot(testLoss,'o-',label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model Loss')

ax[1].plot(trainErr,'s-',label='Train')
ax[1].plot(testErr,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Error (%)')
ax[1].set_title(f'Final model test error rate : {testErr[-1]:.2f}')
ax[1].legend()

plt.show()

In [ ]:
#this experiment takes~8 on a GPU
convChans=[2,5,8]
#initialize results matrix
results=np.zeros((len(convChans),len(convChans),2))
convParams=np.zeros((len(convChans),len(convChans)))
for i,Nchani in enumerate(convChans):
  for j,Nchanj in enumerate(convChans):
    trainLoss,testLoss,trainErr,testErr,net=function2trainTheModel((Nchani,Nchanj))
    #get results
    results[i,j,:]=trainErr[-1],testErr[-1]
    convParams[i,j]=Nchani+Nchanj

    print(i,j)

In [ ]:
#show the results matrix
fig,ax=plt.subplots(1,2,figsize=(10,4))
for i in range(2):
  h=ax[i].imshow(results[:,:,i],vmin=np.min(results),vmax=np.max(results))
  ax[i].set_xlabel('Channel in conv1')
  ax[i].set_ylabel('Channel in conv2')
  ax[i].set_xticks(range(j+1))
  ax[i].set_yticks(range(j+1))
  ax[i].set_xticklabels(convParams)
  ax[i].set_yticklabels(convParams)
  title='Train' if i==0 else 'Test'
  ax[i].set_title('Error rates %s'%title,fontweight='bold')
#add a colorbar right of the plot
axpos=ax[1].get_position()
cax=fig.add_axes([axpos.x1+0.01,axpos.y0,0.01,0.75])
hh=fig.colorbar(h,cax=cax)
hh.set_label('Error rate(%s)',rotation=270,labelpad=10)

plt.show()

In [ ]:
#error rate as a function of the total number of convchannels
corrTrain=np.corrcoef(convParams.flatten(),results[:,:,0].flatten())
corrTest=np.corrcoef(convParams.flatten(),results[:,:,1].flatten())
#plots
plt.plot(convParams.flatten(),results[:,:,0].flatten(),'o-',label=f'Train(r={corrTrain[0,1]:.2f})')
plt.plot(convParams.flatten(),results[:,:,1].flatten(),'o-',label=f'Test(r={corrTest[0,1]:.2f})')
plt.xlabel('Total number of channels')
plt.ylabel('Error rate (%)')
plt.legend()
plt.show()